# Preparing Data for Edinburgh GWR

## EDI NDVI to 10meter polygons

### reading the NDVI raster as polygons 

In [4]:
import geopandas as gpd

In [5]:
ndvi = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_ndvi_clipped.shp")

In [6]:
ndvi.head()

,fid,G,geometry
0,1.0,0.938139,"POLYGON ((257873.463 673070.436, 257873.463 67..."
1,2.0,0.935334,"POLYGON ((257878.802 673071.405, 257893.198 67..."
2,3.0,0.802828,"POLYGON ((256518.858 673042.643, 256536.577 67..."
3,4.0,0.739121,"POLYGON ((256575.897 673026.291, 256575.897 67..."
4,5.0,0.943165,"POLYGON ((257797.299 673005.604, 257794.823 67..."


### joining NDVI with landcover data 

In [ ]:
landcover = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Results\Landcoverc\edi_landcover.shp")
ndvi = gpd.sjoin(ndvi, landcover[["geometry", "lc_class"]], how="left", predicate="intersects")

In [ ]:
gwr_n_lc = ndvi

In [ ]:
gwr_n_lc.head()

In [ ]:
gwr_n_lc["mean_noise"].describe()

In [ ]:
from rasterstats import zonal_stats
stats = zonal_stats(ndvi, r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_sum_10_filled_clipped.tif", stats=["mean"], nodata=None)

In [ ]:
ndvi["mean_noise"] = [s["mean"] for s in stats]

In [ ]:
ndvi.head()

In [17]:
ndvi["mean_noise"].describe()

count    113916.000000
mean          4.574655
std           2.436981
min           0.000000
25%           3.000000
50%           4.836787
75%           6.269535
max          10.000000
Name: mean_noise, dtype: float64

## Trees

In [16]:
import geopandas as gpd
editrees = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\env_data\Trees\Trees.shp")

clip_trees = gpd.clip(editrees, edibounds)

In [ ]:
if "index_right" in gwr_n_lc.columns:
    gwr_n_lc = gwr_n_lc.drop(columns=["index_right"])

In [ ]:
# Here is another way 
tree_counts = gpd.sjoin(clip_trees, gwr_n_lc, how="inner", predicate="within")
tree_summary = tree_counts.groupby("index_right").size()

# Assign counts to the grid
gwr_n_lc["tree_count"] = gwr_n_lc.index.map(tree_summary).fillna(0).astype(int)

In [ ]:
gwr_n_lc.head()

In [ ]:
gwr_n_lc["lc_class"].unique()

In [ ]:
gwr_n_lc["lc_class"] = gwr_n_lc["lc_class"].astype(str)

# Step 2: Filter out rows where lc_class is '0.0' or 'nan'
gwr_n_lc = gwr_n_lc[~gwr_n_lc["lc_class"].isin(["0.0", "nan"])]

In [ ]:
gwr_n_lc["lc_class"].unique()

# Dummy coding 

In [ ]:
gwr_n_lc["lc_class"] = gwr_n_lc["lc_class"].astype("category")

In [ ]:
gwr_n_lc.head()

In [ ]:
import pandas as pd

# Create dummy variables, drop_first=True avoids multicollinearity (reference category)
dummies = pd.get_dummies(gwr_n_lc["lc_class"], prefix="lc", drop_first=True)

# Join them back to the GeoDataFrame
gwr_n_lc_d = gwr_n_lc.join(dummies)

In [ ]:
gwr_n_lc_d["tree_count"].describe()

In [ ]:
gwr_n_lc_d.to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\GWRInput_10m.shp")

In [ ]:
gwr_n_lc_d.info

## centriods 

In [ ]:
# Create a new GeoDataFrame with centroids
centroids_gwr_n_lc_d = gwr_n_lc_d.copy()
centroids_gwr_n_lc_d["geometry"] = gwr_n_lc_d.geometry.centroid

In [ ]:
centroids_gwr_n_lc_d.to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\GWRInput_10m_centriods.shp")

### Creating Urban pct test

In [250]:
noise_gdf["grid_id"] = noise_gdf.index.astype(str)

In [251]:
# testing 
urban_codes = [11100, 11210, 11220, 11230, 11240, 11300, 12100, 12210, 12220, 12230, 12300, 12400, 13100, 13300, 14400]
urban = landcover[landcover["code_2018"].astype(int).isin(urban_codes)]

In [ ]:
urban_overlap = gpd.overlay(noise_gdf, urban, how="intersection")
urban_overlap["urban_area"] = urban_overlap.geometry.area

In [ ]:
urban_by_grid = (
    urban_overlap.groupby("grid_id")["urban_area"]
    .sum()
    .reset_index()
    .rename(columns={"urban_area": "urban_area_m2"})
)

In [ ]:
# Add to full grid
noise_gdf["grid_area"] = noise_gdf.geometry.area  # should be 100 m² for 10x10m
noise_gdf = noise_gdf.merge(urban_by_grid, on="grid_id", how="left")

# Fill NaNs (no urban overlap) with 0
noise_gdf["urban_area_m2"] = noise_gdf["urban_area_m2"].fillna(0)

# Calculate proportion
noise_gdf["urban_pct"] = noise_gdf["urban_area_m2"] / noise_gdf["grid_area"]


### joining with tree count data 

In [197]:
clip_trees.columns

Index(['Z', 'U_ID', 'geometry'], dtype='object')

In [198]:
noise_gdf.columns

Index(['geometry', 'noise', 'index_right', 'code_2018'], dtype='object')

In [199]:
if "index_right" in noise_gdf.columns:
    noise_gdf = noise_gdf.drop(columns=["index_right"])

In [244]:
tree_counts = gpd.sjoin(merged_gdf, noise_gdf, how="inner", predicate="within")
tree_summary = tree_counts.groupby("index_right").size()

# Assign counts to the grid
noise_gdf["tree_count"] = noise_gdf.index.map(tree_summary).fillna(0).astype(int)

KeyboardInterrupt: 

In [ ]:
tree_summary

In [ ]:
noise_gdf.head()

In [214]:
noise_gdf["code_2018"].unique()

array(['23000', '12220', '12100', '21000', '11300', nan, '32000', '31000',
       '14100', '11210', '11220', '14200', '50000', '13400', '13100',
       '13300', '11230', '11240', '12230', '11100', '12210', '12300'],
      dtype=object)

In [220]:
noise_gdf_cleaned = noise_gdf.dropna(subset=["code_2018"])

In [221]:
noise_gdf_cleaned.head()

,geometry,noise,code_2018,tree_count,landcover_group
0,"POLYGON ((257332.254 672874.051, 257332.254 67...",2.0,23000,9,other
1,"POLYGON ((257492.254 672874.051, 257492.254 67...",5.0,23000,1,other
2,"POLYGON ((257502.254 672874.051, 257502.254 67...",4.0,23000,1,other
3,"POLYGON ((257582.254 672874.051, 257582.254 67...",9.0,23000,0,other
4,"POLYGON ((257592.254 672874.051, 257592.254 67...",6.0,23000,0,other


### Reclassifying landcover to urban, green, or forest 

In [28]:
clipped.dtypes

G              float64
geometry      geometry
code_2018       object
mean_noise     float64
tree_count       int32
dtype: object

In [29]:
# Convert to numeric
clipped["code_2018"] = pd.to_numeric(clipped["code_2018"], errors="coerce")

In [30]:
clipped["code_2018"].unique()

array([31000., 23000., 11210., 12220., 11100., 12210., 13400., 13300.,
       14100., 12100., 11220., 11300., 21000., 32000., 11230., 14200.,
       12230., 50000.,    nan, 12300., 13100., 11240.])

In [31]:
import geopandas as gpd
import numpy as np

# Assume your GeoDataFrame is called gdf
def classify_landcover(code):
    if code in [11100, 11210, 11220, 11230, 11240, 11300, 12100, 12210, 12220, 12230, 12300, 12400, 13100, 13300, 14400]:
        return 'urban'  # Urban
    elif code in [14100, 14200, 21000, 22000, 23000, 24000]:
        return 'green'  # Green
    elif code in [25000, 31000, 32000]:
        return 'forest'  # Forest
    elif code == 50000:
        return 'water'  # Water
    else:
        return 'other'  # Or some other default

# Apply classification
clipped["landcover_group"] = clipped["code_2018"].apply(classify_landcover)

In [37]:
clipped.head()

,G,geometry,code_2018,mean_noise,tree_count,landcover_group
112026,0.938214,"POLYGON ((253744.844 658061.653, 253744.844 65...",31000.0,3.849596,8,forest
112025,0.950258,"POLYGON ((253666.204 658061.653, 253705.524 65...",31000.0,4.000000,15,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",31000.0,4.000000,16,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.000000,16,green
111944,0.951493,"POLYGON ((253744.844 658100.973, 253744.844 65...",31000.0,4.000000,29,forest


In [35]:
test = clipped[clipped["landcover_group"] == "green"]
test

,G,geometry,code_2018,mean_noise,tree_count,landcover_group
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.0,16,green
111857,0.957950,"POLYGON ((253705.524 658140.294, 253705.524 65...",23000.0,4.0,4,green
111858,0.955754,"POLYGON ((253744.844 658140.294, 253744.844 65...",23000.0,4.0,30,green
112024,0.959383,"POLYGON ((253626.884 658061.653, 253666.204 65...",23000.0,4.0,6,green
111941,0.961163,"POLYGON ((253626.884 658100.973, 253626.884 65...",23000.0,4.0,0,green
...,...,...,...,...,...,...
15,0.932705,"POLYGON ((257912.784 673003.323, 257912.784 67...",23000.0,0.0,1,green
16,0.914104,"POLYGON ((257952.104 673003.323, 257952.104 67...",23000.0,0.0,0,green
7,0.922904,"POLYGON ((257952.104 673042.643, 257952.104 67...",23000.0,0.0,6,green
6,0.916827,"POLYGON ((257912.784 673042.643, 257912.784 67...",23000.0,0.0,0,green


In [36]:
test["landcover_group"].unique()

array(['green'], dtype=object)

#### Dropping the water category, and all others 

In [38]:
clipped = clipped[~clipped['landcover_group'].isin(['water', 'other'])]

In [39]:
clipped

,G,geometry,code_2018,mean_noise,tree_count,landcover_group
112026,0.938214,"POLYGON ((253744.844 658061.653, 253744.844 65...",31000.0,3.849596,8,forest
112025,0.950258,"POLYGON ((253666.204 658061.653, 253705.524 65...",31000.0,4.000000,15,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",31000.0,4.000000,16,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.000000,16,green
111944,0.951493,"POLYGON ((253744.844 658100.973, 253744.844 65...",31000.0,4.000000,29,forest
...,...,...,...,...,...,...
15,0.932705,"POLYGON ((257912.784 673003.323, 257912.784 67...",23000.0,0.000000,1,green
16,0.914104,"POLYGON ((257952.104 673003.323, 257952.104 67...",23000.0,0.000000,0,green
7,0.922904,"POLYGON ((257952.104 673042.643, 257952.104 67...",23000.0,0.000000,6,green
6,0.916827,"POLYGON ((257912.784 673042.643, 257912.784 67...",23000.0,0.000000,0,green


#### setting the correct ordering 

In [41]:
clipped['landcover_group'] = pd.Categorical(
    clipped['landcover_group'],
    categories=['urban', 'green', 'forest'],
    ordered=True
)
#noise_gdf = pd.get_dummies(noise_gdf, columns=['landcover_group'], drop_first=True)

In [42]:
clipped.head()

,G,geometry,code_2018,mean_noise,tree_count,landcover_group
112026,0.938214,"POLYGON ((253744.844 658061.653, 253744.844 65...",31000.0,3.849596,8,forest
112025,0.950258,"POLYGON ((253666.204 658061.653, 253705.524 65...",31000.0,4.000000,15,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",31000.0,4.000000,16,forest
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.000000,16,green
111944,0.951493,"POLYGON ((253744.844 658100.973, 253744.844 65...",31000.0,4.000000,29,forest


In [ ]:
#noise_gdf = pd.get_dummies(noise_gdf, columns=['landcover_group'], drop_first=True)

## ScALING FEATURES BETWEEN 1 AND 0 

In [43]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
clipped[["noise_norm", "tree_count_norm"]] = scaler.fit_transform(clipped[["mean_noise", "tree_count"]])

In [46]:
clipped.describe()

,G,code_2018,mean_noise,tree_count,noise_norm,tree_count_norm
count,238642.000000,238642.000000,238248.000000,238642.000000,238248.000000,238642.000000
mean,0.772585,13529.364571,5.212250,417.866687,0.521225,0.001472
std,0.128726,4391.632813,2.390296,10496.008650,0.239030,0.036985
min,0.000000,11100.000000,0.000000,0.000000,0.000000,0.000000
25%,0.681115,11220.000000,3.587517,3.000000,0.358752,0.000011
50%,0.776145,12220.000000,5.512898,12.000000,0.551290,0.000042
75%,0.885738,13300.000000,6.934732,24.000000,0.693473,0.000085
max,0.977799,32000.000000,10.000000,283790.000000,1.000000,1.000000


In [47]:
clipped["G"].median()

0.7761445045471191

In [45]:
clipped.crs

<Projected CRS: EPSG:27700>
Name: OSGB36 / British National Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: United Kingdom (UK) - offshore to boundary of UKCS within 49°45'N to 61°N and 9°W to 2°E; onshore Great Britain (England, Wales and Scotland). Isle of Man onshore.
- bounds: (-9.01, 49.75, 2.01, 61.01)
Coordinate Operation:
- name: British National Grid
- method: Transverse Mercator
Datum: Ordnance Survey of Great Britain 1936
- Ellipsoid: Airy 1830
- Prime Meridian: Greenwich

In [48]:
clipped.head()

,G,geometry,code_2018,mean_noise,tree_count,landcover_group,noise_norm,tree_count_norm
112026,0.938214,"POLYGON ((253744.844 658061.653, 253744.844 65...",31000.0,3.849596,8,forest,0.38496,0.000028
112025,0.950258,"POLYGON ((253666.204 658061.653, 253705.524 65...",31000.0,4.000000,15,forest,0.40000,0.000053
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",31000.0,4.000000,16,forest,0.40000,0.000056
111943,0.954568,"POLYGON ((253705.524 658100.973, 253705.524 65...",23000.0,4.000000,16,green,0.40000,0.000056
111944,0.951493,"POLYGON ((253744.844 658100.973, 253744.844 65...",31000.0,4.000000,29,forest,0.40000,0.000102


In [50]:
clipped.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_1m.shp")

C:\Users\ibk1\AppData\Local\Temp\ipykernel_18376\2400969659.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  clipped.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_1m.shp")
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'landcover_group' to 'landcover_'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'tree_count_norm' to 'tree_cou_1'
  ogr_write(


# Filling in no data values 

In [52]:
import rasterio
import numpy as np

# Load the raster
with rasterio.open("C:/Users/ibk1/NoiseModelling/Glasgow/Analysis/summer/gg_sim_1m_clipped.tif") as src:
    noise = src.read(1)
    profile = src.profile
    nodata = src.nodata or np.nan


In [53]:
# Create a mask of missing values
mask = np.isnan(noise)


In [54]:
from scipy.ndimage import generic_filter

# Function to compute mean of non-NaNs
def nanmean_filter(values):
    return np.nanmean(values)

# Apply focal mean (kernel size = 7 → 70m across if 10m resolution)
kernel_size = 7  # Adjust depending on your radius
smoothed = generic_filter(noise, nanmean_filter, size=kernel_size, mode='constant', cval=np.nan)

In [ ]:
# Correct mask
mask = (noise == nodata) if nodata is not None else np.isnan(noise)

# Smoothing kernel (e.g., 7x7 ~ 70m window)
def nanmean_filter(values):
    return np.nanmean(values)

smoothed = generic_filter(noise.astype(float), nanmean_filter, size=7, mode='constant', cval=np.nan)

# Fill missing areas
filled = np.where(mask, smoothed, noise)
profile.update(dtype=rasterio.float32, nodata=None)

In [55]:
filled = np.where(mask, smoothed, noise)

In [56]:
with rasterio.open("C:/Users/ibk1/NoiseModelling/Glasgow/Analysis/summer/gg_sum_1m_filled.tif", "w", **profile) as dst:
    dst.write(filled, 1)

# GWR Preprocessing by 50x50 meter grid

In [27]:
ndns_edi = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\ndns_edi.shp")

In [28]:
ndns_edi.head()

,id,left,top,right,bottom,row_index,col_index,noise_mean,ndvi_mean,geometry
0,116.0,309581.27,674079.4388,309631.27,674029.4388,115.0,0.0,5.940886,0.750596,"POLYGON ((309581.27 674049.75, 309624.147 6740..."
1,117.0,309581.27,674029.4388,309631.27,673979.4388,116.0,0.0,5.947170,0.797000,"POLYGON ((309615.48 673996.245, 309594.257 674..."
2,118.0,309581.27,673979.4388,309631.27,673929.4388,117.0,0.0,6.000000,0.635603,"POLYGON ((309625.667 673979.439, 309631.27 673..."
3,528.0,309631.27,674079.4388,309681.27,674029.4388,115.0,1.0,5.903782,0.674871,"POLYGON ((309664.793 674067.189, 309681.27 674..."
4,529.0,309631.27,674029.4388,309681.27,673979.4388,116.0,1.0,5.488000,0.688496,"POLYGON ((309631.27 674029.439, 309681.27 6740..."


In [32]:
from rasterstats import zonal_stats
import pandas as pd

In [30]:
stats = zonal_stats(ndns_edi, r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\edi_cluster analysis\lc_cls_v2_50by50_raster.tif", categorical=True)

In [41]:
stats_df = pd.DataFrame(stats).fillna(0)  # fill NaN with 0 where class is missing
stats_df.columns = [f'class_{int(col)}' for col in stats_df.columns]

ndns_edi = ndns_edi.reset_index(drop=True)  # Ensure same index
ndns_edi = pd.concat([ndns_edi, stats_df], axis=1)


In [42]:
ndns_edi.columns

Index([        'id',       'left',        'top',      'right',     'bottom',
        'row_index',  'col_index', 'noise_mean',  'ndvi_mean',   'geometry',
                1.0,          2.0,          3.0,  'total_pix',    'class_1',
          'class_2',    'class_3'],
      dtype='object')

In [43]:
# Total pixels (can also use sum of class columns)
ndns_edi['total_pix'] = stats_df.sum(axis=1)

# Example: forest (class 3) percentage
ndns_edi['forest_pct'] = ndns_edi['class_3'] / ndns_edi['total_pix']
ndns_edi['grass_pct'] = ndns_edi['class_2'] / ndns_edi['total_pix']
ndns_edi['urban_pct'] = ndns_edi['class_1'] / ndns_edi['total_pix']


In [49]:
ndns_edi.columns

Index([        'id',       'left',        'top',      'right',     'bottom',
        'row_index',  'col_index', 'noise_mean',  'ndvi_mean',   'geometry',
                1.0,          2.0,          3.0,  'total_pix',    'class_1',
          'class_2',    'class_3', 'forest_pct',  'grass_pct',  'urban_pct'],
      dtype='object')

In [ ]:
ndns_edi = ndns_edi.drop(columns=['total_pix',
                1.0,          2.0,          3.0,])

In [53]:
ndns_edi.head()

,id,left,top,right,bottom,row_index,col_index,noise_mean,ndvi_mean,geometry,class_1,class_2,class_3,forest_pct,grass_pct,urban_pct
0,116.0,309581.27,674079.4388,309631.27,674029.4388,115.0,0.0,5.940886,0.750596,"POLYGON ((309581.27 674049.75, 309624.147 6740...",1.0,0.0,0.0,0.0,0.0,1.0
1,117.0,309581.27,674029.4388,309631.27,673979.4388,116.0,0.0,5.947170,0.797000,"POLYGON ((309615.48 673996.245, 309594.257 674...",0.0,0.0,0.0,NaN,NaN,NaN
2,118.0,309581.27,673979.4388,309631.27,673929.4388,117.0,0.0,6.000000,0.635603,"POLYGON ((309625.667 673979.439, 309631.27 673...",0.0,0.0,0.0,NaN,NaN,NaN
3,528.0,309631.27,674079.4388,309681.27,674029.4388,115.0,1.0,5.903782,0.674871,"POLYGON ((309664.793 674067.189, 309681.27 674...",0.0,1.0,0.0,0.0,1.0,0.0
4,529.0,309631.27,674029.4388,309681.27,673979.4388,116.0,1.0,5.488000,0.688496,"POLYGON ((309631.27 674029.439, 309681.27 6740...",0.0,1.0,0.0,0.0,1.0,0.0


In [13]:
# Each dict in `stats` will have keys like: {1: count, 2: count, 3: count}
# Add them to your GeoDataFrame
ndns_edi["urban_count"] = [s.get(1, 0) for s in stats]
ndns_edi["green_count"] = [s.get(2, 0) for s in stats]
ndns_edi["forest_count"] = [s.get(3, 0) for s in stats]

In [18]:
ndns_edi["total"] = ndns_edi[["urban_count", "green_count", "forest_count"]].sum(axis=1)
ndns_edi["urban_share"] = ndns_edi["urban_count"] / ndns_edi["total"]

In [19]:
from geopandas.tools import sjoin

# Perform the spatial join only once, with how="left"
joined = sjoin(ndns_edi, clip_trees, how="left")

# Now group by the index of the left GeoDataFrame (ndns)
tree_counts = joined.groupby(joined.index).size()

# Add counts to the original GeoDataFrame
ndns_edi["tree_count"] = ndns_edi.index.map(tree_counts).fillna(0).astype(int)

In [20]:
ndns_edi["x"] = ndns_edi.geometry.centroid.x
ndns_edi["y"] = ndns_edi.geometry.centroid.y

In [23]:
ndns_edi.head(1)

,id,left,top,right,bottom,row_index,col_index,noise_mean,ndvi_mean,geometry,urban_count,green_count,forest_count,total,urban_share,tree_count,x,y
0,116.0,309581.27,674079.4388,309631.27,674029.4388,115.0,0.0,5.940886,0.750596,"POLYGON ((309581.27 674049.75, 309624.147 6740...",0,0,0,0,NaN,1,309610.635189,674043.430571


In [25]:
ndns_edi["urban_count"].unique()

array([0, 1], dtype=int64)

In [27]:
ndns_edi[["geometry","x", "y", "noise_mean", "ndvi_mean", "tree_count", "urban_count", "green_count", "forest_count"]].to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_gwr_input_50m.shp", index=False)

C:\Users\ibk1\AppData\Local\Temp\ipykernel_21080\279223687.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  ndns_edi[["geometry","x", "y", "noise_mean", "ndvi_mean", "tree_count", "urban_count", "green_count", "forest_count"]].to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_gwr_input_50m.shp", index=False)
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'urban_count' to 'urban_coun'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'green_count' to 'green_coun'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'forest_count' to 'forest_cou'
  ogr_write(


In [31]:
ndns_edi.columns

Index(['id', 'left', 'top', 'right', 'bottom', 'row_index', 'col_index',
       'noise_mean', 'ndvi_mean', 'geometry', 'urban_count', 'green_count',
       'forest_count', 'total', 'urban_share', 'tree_count', 'x', 'y'],
      dtype='object')

In [32]:
ndns_edi.isna().sum()

id                 0
left               0
top                0
right              0
bottom             0
row_index          0
col_index          0
noise_mean       358
ndvi_mean          4
geometry           0
urban_count        0
green_count        0
forest_count       0
total              0
urban_share     6697
tree_count         0
x                  0
y                  0
dtype: int64

In [33]:
ndns_edi_cleaned = ndns_edi.dropna(subset=['noise_mean'])

In [37]:
ndns_edi_cleaned.isna().sum()

id              0
left            0
top             0
right           0
bottom          0
row_index       0
col_index       0
noise_mean      0
ndvi_mean       0
geometry        0
urban_count     0
green_count     0
forest_count    0
total           0
urban_share     0
tree_count      0
x               0
y               0
dtype: int64

In [35]:
len(ndns_edi_cleaned)

110187

In [36]:
# Replace NaN values in urban_share with 0
ndns_edi_cleaned["urban_share"] = ndns_edi_cleaned["urban_share"].fillna(0)

C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [38]:
ndns_edi_cleaned.head()

,id,left,top,right,bottom,row_index,col_index,noise_mean,ndvi_mean,geometry,urban_count,green_count,forest_count,total,urban_share,tree_count,x,y
0,116.0,309581.27,674079.4388,309631.27,674029.4388,115.0,0.0,5.940886,0.750596,"POLYGON ((309581.27 674049.75, 309624.147 6740...",0,0,0,0,0.0,1,309610.635189,674043.430571
1,117.0,309581.27,674029.4388,309631.27,673979.4388,116.0,0.0,5.947170,0.797000,"POLYGON ((309615.48 673996.245, 309594.257 674...",0,0,0,0,0.0,1,309618.747028,674010.658740
2,118.0,309581.27,673979.4388,309631.27,673929.4388,117.0,0.0,6.000000,0.635603,"POLYGON ((309625.667 673979.439, 309631.27 673...",0,0,0,0,0.0,1,309629.402336,673976.357635
3,528.0,309631.27,674079.4388,309681.27,674029.4388,115.0,1.0,5.903782,0.674871,"POLYGON ((309664.793 674067.189, 309681.27 674...",1,0,0,1,1.0,1,309657.089062,674047.361718
4,529.0,309631.27,674029.4388,309681.27,673979.4388,116.0,1.0,5.488000,0.688496,"POLYGON ((309631.27 674029.439, 309681.27 6740...",0,1,0,1,0.0,1,309656.270000,674004.438800


In [39]:
ndns_edi_cleaned[["geometry","x", "y", "noise_mean", "ndvi_mean", "tree_count", "urban_count", "green_count", "forest_count"]].to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_gwr_input_50m_cleaned.shp", index=False)

C:\Users\ibk1\AppData\Local\Temp\ipykernel_21080\1772665577.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  ndns_edi_cleaned[["geometry","x", "y", "noise_mean", "ndvi_mean", "tree_count", "urban_count", "green_count", "forest_count"]].to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_gwr_input_50m_cleaned.shp", index=False)
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'urban_count' to 'urban_coun'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'green_count' to 'green_coun'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'forest_count' to 'forest_cou'
  ogr_write(


In [40]:
dz_bounds = gpd.read_file(r"C:/Users/ibk1/NoiseModelling/Edinburgh/Boundary/datazone_bounds_te.shp")

In [41]:
dz_bounds.explore()

In [42]:
ndns_edi_cleaned_clipped = gpd.clip(ndns_edi_cleaned, dz_bounds)

In [43]:
ndns_edi_cleaned_clipped[["geometry","x", "y", "noise_mean", "ndvi_mean", "tree_count", "urban_count", "green_count", "forest_count"]].to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_gwr_input_50m_cleaned_clipped.shp", index=False)

C:\Users\ibk1\AppData\Local\Temp\ipykernel_21080\624133469.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  ndns_edi_cleaned_clipped[["geometry","x", "y", "noise_mean", "ndvi_mean", "tree_count", "urban_count", "green_count", "forest_count"]].to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_gwr_input_50m_cleaned_clipped.shp", index=False)
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'urban_count' to 'urban_coun'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'green_count' to 'green_coun'
  ogr_write(
C:\Users\ibk1\AppData\Local\miniconda3\envs\noiseprop\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'forest_count' to 'forest_cou'
  ogr_write(


In [46]:
ndns_edi_cleaned_clipped.columns

Index(['id', 'left', 'top', 'right', 'bottom', 'row_index', 'col_index',
       'noise_mean', 'ndvi_mean', 'geometry', 'urban_count', 'green_count',
       'forest_count', 'total', 'urban_share', 'tree_count', 'x', 'y'],
      dtype='object')

# GWR Processing by Datazones

In [1]:
import geopandas as gpd

In [4]:
# Reading the datazones shapefile 
edidz = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\env_data\TE\edi\edi_te_final.shp")
# running raster stats 
stats = zonal_stats(edidz, r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_sum1m_rast_filled.tif")

In [5]:
edidz["mean_noise"] = [s["mean"] for s in stats]

In [6]:
edidz.columns

Index(['bge_code', 'bge_type', 'la_name', 'co_name', 'reg_name', 'country',
       'total_pop', 'urban_area', 'urban_pct', 'tc_goal', 'treecanopy',
       'tc_gap', 'priority_i', 'inc_rank', 'incnorm', 'inc_dec', 'emp_rank',
       'empnorm', 'emp_dec', 'hlth_rank', 'hlthnorm', 'hlth_dec', 'temp_diff',
       'tempnorm', 'NO2_avg', 'PM25_avg', 'apb_index', 'dep_ratio',
       'depratnorm', 'dep_perc', 'tes', 'la_tes', 'pctmineth', 'pct_child',
       'pct_senior', 'la_code', 'co_code', 'peat', 'geometry', 'mean_noise'],
      dtype='object')

In [8]:
edidz["apb_index"].head()

0    0.211568
1    0.223798
2    0.246258
3    0.229363
4    0.251449
Name: apb_index, dtype: float64

In [9]:
SIMD = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\SIMD Data\SG_SIMD_2020.shp")

In [12]:
SIMD.columns

Index(['DataZone', 'DZName', 'LAName', 'SAPE2017', 'WAPE2017', 'Rankv2',
       'Quintilev2', 'Decilev2', 'Vigintilv2', 'Percentv2', 'IncRate',
       'IncNumDep', 'IncRankv2', 'EmpRate', 'EmpNumDep', 'EmpRank', 'HlthCIF',
       'HlthAlcSR', 'HlthDrugSR', 'HlthSMR', 'HlthDprsPc', 'HlthLBWTPc',
       'HlthEmergS', 'HlthRank', 'EduAttend', 'EduAttain', 'EduNoQuals',
       'EduPartici', 'EduUniver', 'EduRank', 'GAccPetrol', 'GAccDTGP',
       'GAccDTPost', 'GAccDTPsch', 'GAccDTSsch', 'GAccDTRet', 'GAccPTGP',
       'GAccPTPost', 'GAccPTRet', 'GAccBrdbnd', 'GAccRank', 'CrimeCount',
       'CrimeRate', 'CrimeRank', 'HouseNumOC', 'HouseNumNC', 'HouseOCrat',
       'HouseNCrat', 'HouseRank', 'Shape_Leng', 'Shape_Area', 'geometry'],
      dtype='object')

In [14]:
SIMD["LAName"].unique()

array(['Aberdeen City', 'Aberdeenshire', 'Angus', 'Argyll and Bute',
       'Clackmannanshire', 'Dumfries and Galloway', 'Dundee City',
       'East Ayrshire', 'East Dunbartonshire', 'East Lothian',
       'East Renfrewshire', 'City of Edinburgh', 'Na h-Eileanan an Iar',
       'Falkirk', 'Fife', 'Glasgow City', 'Highland', 'Inverclyde',
       'Midlothian', 'Moray', 'North Ayrshire', 'North Lanarkshire',
       'Orkney Islands', 'Perth and Kinross', 'Renfrewshire',
       'Scottish Borders', 'Shetland Islands', 'South Ayrshire',
       'South Lanarkshire', 'Stirling', 'West Dunbartonshire',
       'West Lothian'], dtype=object)

In [15]:
ediSIMD = SIMD[SIMD["LAName"] == "City of Edinburgh"]

In [23]:
ediSIMD.columns

Index(['DataZone', 'DZName', 'LAName', 'SAPE2017', 'WAPE2017', 'Rankv2',
       'Quintilev2', 'Decilev2', 'Vigintilv2', 'Percentv2', 'IncRate',
       'IncNumDep', 'IncRankv2', 'EmpRate', 'EmpNumDep', 'EmpRank', 'HlthCIF',
       'HlthAlcSR', 'HlthDrugSR', 'HlthSMR', 'HlthDprsPc', 'HlthLBWTPc',
       'HlthEmergS', 'HlthRank', 'EduAttend', 'EduAttain', 'EduNoQuals',
       'EduPartici', 'EduUniver', 'EduRank', 'GAccPetrol', 'GAccDTGP',
       'GAccDTPost', 'GAccDTPsch', 'GAccDTSsch', 'GAccDTRet', 'GAccPTGP',
       'GAccPTPost', 'GAccPTRet', 'GAccBrdbnd', 'GAccRank', 'CrimeCount',
       'CrimeRate', 'CrimeRank', 'HouseNumOC', 'HouseNumNC', 'HouseOCrat',
       'HouseNCrat', 'HouseRank', 'Shape_Leng', 'Shape_Area', 'geometry'],
      dtype='object')

In [24]:
cols_to_keep = ['DataZone','HlthCIF',
       'HlthAlcSR', 'HlthDrugSR', 'HlthSMR', 'HlthDprsPc', 'HlthLBWTPc',
       'HlthEmergS', 'HlthRank','HouseNumOC', 'HouseNumNC', 'HouseOCrat',
       'HouseNCrat', 'HouseRank', "Vigintilv2",'EduRank', "GAccRank", "CrimeRate"
               ]
merged = edidz.merge(ediSIMD[cols_to_keep], left_on="bge_code", right_on="DataZone", how="left")

In [25]:
# Remove % and convert to float
merged["HouseOCrat"] = merged["HouseOCrat"].str.replace("%", "").astype(float)
merged["HouseNCrat"] = merged["HouseNCrat"].str.replace("%", "").astype(float)
merged["HlthDprsPc"] = merged["HlthDprsPc"].str.replace("%", "").astype(float)
merged["HlthLBWTPc"] = merged["HlthLBWTPc"].str.replace("%", "").astype(float)
# Optional: Convert to decimals (0.1 instead of 10)
# ggsimd["HouseOCrat"] /= 100
# ggsimd["HouseNCrat"] /= 100

In [26]:
merged.to_file(r"C:\Users\ibk1\NoiseModelling\Edinburgh\Analysis\summer\edi_GWR_Input_summer_datazones_SIMD_extra_final.shp")